# Build Mistral 7B from scratch

Plain PyTorch, no transformers library. Tokenizer → RoPE → grouped-query attention → KV cache → transformer → greedy decoding, loading the real Mistral 7B v0.3 weights.

Run every section in order. Full model needs a GPU with ~24 GB (Colab L4). Weights are 14.5 GB — keep them on Drive.

## 0. Setup

Install, imports, and the two config values everything else reads.

In [ ]:
!pip install mistral_common safetensors simple_parsing

In [ ]:
import torch
import os
import json
import safetensors.torch
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Tuple
from simple_parsing.helpers import Serializable
from torch import nn
from mistral_common.tokens.tokenizers.base import Tokenizer
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer

In [ ]:
# change WEIGHTS_DIR if you keep weights elsewhere
WEIGHTS_DIR = "/content/drive/MyDrive/mistral-build/weights"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(DEVICE)

Mount Drive so the weights survive a runtime restart. Harmless outside Colab.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("not on Colab, skipping drive mount")

## 1. Download weights

Official Mistral 7B v0.3 tarball into `WEIGHTS_DIR`. Skips if already there. If the CDN link ever dies, the same three files are on Hugging Face at `mistralai/Mistral-7B-v0.3`.

In [ ]:
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if os.path.exists(f"{WEIGHTS_DIR}/consolidated.safetensors"):
    print("weights already there, skipping download")
else:
    !wget -O "{WEIGHTS_DIR}/mistral-7B-v0.3.tar" https://models.mistralcdn.com/mistral-7b-v0-3/mistral-7B-v0.3.tar
    !tar -xvf "{WEIGHTS_DIR}/mistral-7B-v0.3.tar" -C "{WEIGHTS_DIR}"
    !rm "{WEIGHTS_DIR}/mistral-7B-v0.3.tar"

expected = ["consolidated.safetensors", "params.json", "tokenizer.model.v3"]
missing = [f for f in expected if not os.path.exists(f"{WEIGHTS_DIR}/{f}")]
assert not missing, f"missing in {WEIGHTS_DIR}: {missing}"

## 2. Tokenization

v3 SentencePiece tokenizer. `bos=True` prepends token 1 (`<s>`).

In [ ]:
mistral_tokenizer = MistralTokenizer.from_file(f"{WEIGHTS_DIR}/tokenizer.model.v3")
tokenizer = mistral_tokenizer.instruct_tokenizer.tokenizer

In [ ]:
prompts = [
    "Who is the greatest tennis player of all time",
    "Explain rotary positional embeddings in one line",
]
encoded_prompts = [tokenizer.encode(prompt, bos=True, eos=False) for prompt in prompts]
for p in encoded_prompts:
    print(p)

## 3. Model config

Mistral 7B v0.3 hyperparameters. 32 query heads over 8 KV heads is the GQA ratio. `sliding_window` is W from the paper — v0.3's `params.json` doesn't set one, so the paper's 4096 is the default.

In [ ]:
@dataclass
class TransformerArgs(Serializable):
    dim: int = 4096
    n_layers: int = 32
    head_dim: int = 128
    hidden_dim: int = 14336
    n_heads: int = 32
    n_kv_heads: int = 8
    norm_eps: float = 1e-05
    vocab_size: int = 32768

    # params.json calls this rope_theta, simple_parsing drops it with a warning - expected, default matches
    rope_N: Optional[float] = 1000000.0

    max_seq_len: int = 16384
    # W from the paper. token i attends to j where i - W < j <= i. v0.3 ships no value, 4096 is the paper's
    sliding_window: int = 4096
    max_batch_size: int = 1

In [ ]:
args = TransformerArgs()

## 4. Rotary positional encoding

RoPE pairs neighbouring dims of each head and rotates pair *i* at position *m* by `m * theta_i`. Position becomes relative: attention between two tokens depends only on their distance. cos/sin are precomputed once for every position.

In [ ]:
# head_dim 8:  x1 x2 x3 x4 x5 x6 x7 x8  ->  pairs (x1,x2) (x3,x4) (x5,x6) (x7,x8), one 2-D rotation each

In [ ]:
def precompute_thetas(dim, end, rope_N):
  thetas = 1.0/(rope_N**(torch.arange(0, dim, 2)[:(dim//2)].float()/dim))
  m_thetas = torch.outer(torch.arange(0, end), thetas)
  complexthetas = torch.polar(torch.ones_like(m_thetas), m_thetas)
  cosin_m_thetas = torch.stack([complexthetas.real, complexthetas.imag], dim=-1)
  return cosin_m_thetas

In [ ]:
def apply_rotary_emb(x, cosin_m_thetas):
  # x: (bs, seqlen, n_heads, head_dim), cosin_m_thetas: (seqlen, head_dim/2, 2)
  xreshaped = x.float().reshape(*x.shape[:-1], -1, 2)

  cosin_m_thetas = cosin_m_thetas.view(1, xreshaped.size(1), 1, xreshaped.size(3), 2)

  x_rot = torch.stack(
        [
            xreshaped[..., 0] * cosin_m_thetas[..., 0] - xreshaped[..., 1] * cosin_m_thetas[..., 1],
            xreshaped[..., 1] * cosin_m_thetas[..., 0] + xreshaped[..., 0] * cosin_m_thetas[..., 1],
        ],
        -1,
  )
  x_rot = x_rot.flatten(3)
  return x_rot.type_as(x)

In [ ]:
# tiny demo: dim=8 -> 4 thetas, 2 positions. position 0 = no rotation (cos 1, sin 0)
dim, end = 8, 2
thetas = 1.0/(args.rope_N**(torch.arange(0, dim, 2)[:(dim//2)].float()/dim))
m_thetas = torch.outer(torch.arange(0, end), thetas)
print("thetas", thetas)
print("m_thetas", m_thetas)
print("cos/sin", precompute_thetas(dim, end, args.rope_N))

In [ ]:
cosin_m_thetas = precompute_thetas(128, 5, args.rope_N)
print(cosin_m_thetas.shape)
print(apply_rotary_emb(torch.randn(1, 5, 32, 128), cosin_m_thetas).shape)

## 5. Grouped query attention

32 query heads share 8 KV heads. `repeat_kv` copies each KV head `n_heads // n_kv_heads` times so the matmul lines up. Smaller KV cache, nearly MHA quality.

In [ ]:
def repeat_kv(keys: torch.Tensor, values: torch.Tensor, repeats: int) -> Tuple[torch.Tensor]:
    keys = torch.repeat_interleave(keys, repeats=repeats, dim=2)
    values = torch.repeat_interleave(values, repeats=repeats, dim=2)
    return keys, values

## 6. KV cache

The cache is a preallocated `(batch, max_seq_len, n_kv_heads, head_dim)` buffer. `scatter_` writes each token's K/V into its position slot, so decode never recomputes old keys. Toy version: 4 slots, 3 tokens.

In [ ]:
c_k = torch.zeros((1, 4, 2, 2), dtype=torch.float16)

x_k = torch.tensor([[
    [[3, 4], [5, 6]],
    [[13, 14], [15, 16]],
    [[13, 14], [15, 16]],
]])
print(x_k.shape)  # (1, 3, 2, 2)

In [ ]:
positions = torch.arange(0, 3)
scatter_pos = positions[None, :, None, None].repeat(1, 1, 2, 2)
print(scatter_pos, scatter_pos.shape)

In [ ]:
c_k = c_k.type(x_k.dtype)  # scatter_ needs matching dtypes
c_k[:1].scatter_(dim=1, index=scatter_pos, src=x_k)
print(c_k)

## 7. Attention

Q/K/V projections → RoPE on Q and K → write K/V to cache → scaled dot-product attention → output projection. Prefill (many positions) uses the fresh K/V; decode (one position) reads everything so far from the cache.

In [ ]:
class Attention(nn.Module):
  def __init__(self, args: TransformerArgs):
    super().__init__()
    self.args = args

    self.n_heads: int = args.n_heads
    self.n_kv_heads: int = args.n_kv_heads

    self.repeats = self.n_heads // self.n_kv_heads
    self.scale = self.args.head_dim**-0.5

    self.wq = nn.Linear(args.dim, args.n_heads * args.head_dim, bias=False)
    self.wk = nn.Linear(args.dim, args.n_kv_heads * args.head_dim, bias=False)
    self.wv = nn.Linear(args.dim, args.n_kv_heads * args.head_dim, bias=False)

    self.wo = nn.Linear(args.n_heads * args.head_dim, args.dim, bias=False)

    # plain tensors, not buffers - put them on DEVICE ourselves
    self.cache_k = torch.empty(
        (
            args.max_batch_size,
            args.max_seq_len,
            self.n_kv_heads,
            self.args.head_dim,
        ),
        dtype = torch.float16,
    ).to(DEVICE)
    self.cache_v = torch.empty(
        (
            args.max_batch_size,
            args.max_seq_len,
            self.n_kv_heads,
            self.args.head_dim,
        ),
        dtype = torch.float16,
    ).to(DEVICE)

  def forward(
        self,
        x: torch.Tensor,
        cosin_m_thetas: torch.Tensor,
        positions: torch.Tensor,
        mask: Optional[torch.Tensor],
      ) -> torch.Tensor:

      bsz,seqlen,_ = x.shape
      xq,xk,xv = self.wq(x), self.wk(x), self.wv(x)

      xq = xq.view(bsz, seqlen, self.n_heads, self.args.head_dim)
      xk = xk.view(bsz, seqlen, self.n_kv_heads, self.args.head_dim)
      xv = xv.view(bsz, seqlen, self.n_kv_heads, self.args.head_dim)

      xq = apply_rotary_emb(xq, cosin_m_thetas)
      xk = apply_rotary_emb(xk, cosin_m_thetas)

      scatter_pos = positions[None, :, None, None].repeat(bsz, 1,self.n_kv_heads, self.args.head_dim)

      self.cache_k = self.cache_k.type(xk.dtype)
      self.cache_v = self.cache_v.type(xv.dtype)

      self.cache_k[:bsz].scatter_(dim=1, index=scatter_pos, src=xk)
      self.cache_v[:bsz].scatter_(dim=1, index=scatter_pos, src=xv)

      if positions.shape[0] > 1:
            # prefill
            key, value = repeat_kv(xk, xv, self.repeats)
      else:
            # decode - read everything so far from the cache
            cur_pos = positions[-1].item() + 1
            key, value = repeat_kv(
                self.cache_k[:bsz, :cur_pos, ...],
                self.cache_v[:bsz, :cur_pos, ...],
                self.repeats,
            )

      # (bsz, seqlen, heads, head_dim) -> (bsz, heads, seqlen, head_dim)
      query = xq.transpose(1,2)
      key = key.transpose(1,2)
      value = value.transpose(1,2)

      scores = torch.matmul(query, key.transpose(2,3))*self.scale
      if mask is not None:
        scores += mask[None,None,...]
      scores = nn.functional.softmax(scores, dim=-1).type_as(query)
      output = torch.matmul(scores, value)

      output = output.transpose(1,2).contiguous().view(bsz,seqlen,-1)
      return self.wo(output)

## 8. Attention smoke test

One attention layer, random weights, real tokenized prompt. Full-size args — GPU.

In [ ]:
prompts = [
    "Who is the greatest tennis player of all time"
]
encoded_prompts = [tokenizer.encode(prompt, bos=True, eos=False) for prompt in prompts]
input_ids = torch.tensor(encoded_prompts).to(DEVICE)
inp_len = input_ids.shape[1]
positions = torch.arange(0, inp_len).to(DEVICE)
print(input_ids, positions)

In [ ]:
tok_embeddings = nn.Embedding(args.vocab_size, args.dim).to(DEVICE)
x = tok_embeddings(input_ids)

cosin_m_thetas = precompute_thetas(args.head_dim, inp_len, args.rope_N).to(DEVICE)
print(cosin_m_thetas.shape)

attention = Attention(args).to(DEVICE)
with torch.no_grad():
    att_out = attention(x, cosin_m_thetas, positions, None)  # shapes only, no mask
print(att_out.shape)

## 9. FeedForward, RMSNorm, TransformerBlock

SwiGLU MLP (`w2(silu(w1 x) * w3 x)`), RMSNorm (no mean subtraction), and the pre-norm block with two residual connections.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, args: TransformerArgs):
        super().__init__()

        self.w1 = nn.Linear(args.dim, args.hidden_dim, bias=False)
        self.w2 = nn.Linear(args.hidden_dim, args.dim, bias=False)
        self.w3 = nn.Linear(args.dim, args.hidden_dim, bias=False)

    def forward(self, x) -> torch.Tensor:
        return self.w2(nn.functional.silu(self.w1(x)) * self.w3(x))


class RMSNorm(torch.nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        output = self._norm(x.float()).type_as(x)
        return output * self.weight


class TransformerBlock(nn.Module):
    def __init__(self, args: TransformerArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.dim = args.dim
        self.attention = Attention(args)
        self.feed_forward = FeedForward(args=args)
        self.attention_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.ffn_norm = RMSNorm(args.dim, eps=args.norm_eps)
        self.args = args

    def forward(
        self,
        x: torch.Tensor,
        cosin_m_thetas: torch.Tensor,
        positions: torch.Tensor,
        mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        r = self.attention.forward(self.attention_norm(x), cosin_m_thetas, positions, mask)
        h = x + r
        r = self.feed_forward.forward(self.ffn_norm(h))
        out = h + r
        return out

## 10. Transformer

Embeddings → 32 blocks → final RMSNorm → LM head. The causal mask is built only on prefill; a single decode token has nothing to mask. `from_folder` reads `params.json` and loads the safetensors checkpoint.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, args: TransformerArgs):
        super().__init__()
        self.args = args
        self.vocab_size = args.vocab_size
        self.n_layers = args.n_layers
        assert self.vocab_size > 0

        self.tok_embeddings = nn.Embedding(args.vocab_size, args.dim)

        self.layers = torch.nn.ModuleList([TransformerBlock(args=args) for _ in range(args.n_layers)])

        self.norm = RMSNorm(args.dim, eps=args.norm_eps)

        self.output = nn.Linear(args.dim, args.vocab_size, bias=False)

        theta = self.args.rope_N
        self.cosin_m_thetas = precompute_thetas(self.args.head_dim, 128_000, theta).to(DEVICE)

    def forward(
        self,
        input_ids: torch.Tensor,
        positions: torch.Tensor,
    ):
        h = self.tok_embeddings(input_ids)
        cosin_m_thetas = self.cosin_m_thetas[positions]
        mask: Optional[torch.Tensor] = None
        if input_ids.shape[1] > 1:
            # causal mask, only needed on prefill
            seqlen = input_ids.shape[1]
            tensor = torch.full(
                (seqlen, seqlen),
                dtype=h.dtype,
                fill_value=1,
                device=h.device,
            )
            mask = torch.tril(tensor, diagonal=0).to(h.dtype)
            mask = torch.triu(mask, diagonal=-(self.args.sliding_window - 1))  # drop anything older than W
            mask = torch.log(mask)

        for i,layer in enumerate(self.layers):
            h = layer(h, cosin_m_thetas, positions, mask)

        return self.output(self.norm(h)).float()

    @staticmethod
    def from_folder(folder: Path, max_batch_size: int = 1, device=DEVICE, dtype=torch.float16):
        with open(Path(folder) / "params.json", "r") as f:
            model_args = TransformerArgs.from_dict(json.load(f))
        model_args.max_batch_size = max_batch_size

        model = Transformer(model_args)

        pt_model_file = Path(folder) / "consolidated.00.pth"
        safetensors_model_file = Path(folder) / "consolidated.safetensors"

        if pt_model_file.exists():
            loaded = torch.load(str(pt_model_file), mmap=True)
        else:
            loaded = safetensors.torch.load_file(str(safetensors_model_file))

        model.load_state_dict(loaded, assign=True, strict=True)

        return model.to(device=device, dtype=dtype)

## 11. Causal mask

What `forward` builds. Token *i* may attend to token *j* only if

$$ i - W < j \le i $$

`tril` gives the causal half ($j \le i$), `triu` with `diagonal=-(W-1)` cuts everything older than W, `log` turns 1 → 0 (attend) and 0 → −inf (blocked). Each layer sees W tokens back; after *k* layers a token's information has travelled *k·W* positions — that's how a 4096 window covers a 32k context.

In [ ]:
W = 3  # tiny window so you can see it. the model uses args.sliding_window
m = torch.tril(torch.full((6, 6), fill_value=1.0), diagonal=0)
m = torch.triu(m, diagonal=-(W - 1))
print(torch.log(m))

## 12. Generate

Prefill the whole prompt in one forward pass, then greedy-decode one token at a time through the KV cache. Loads the 7B weights — first call takes a couple of minutes.

The window is enforced on prefill. Decode reads the whole cache, which is fine while prompt + output stay under W. Making decode roll too — a cache of W slots written at `position % W` — is the paper's rolling buffer, and the exercise if you want to go further.

In [ ]:
@torch.no_grad()
def generate(prompts: List[str], model: Transformer, tokenizer: Tokenizer, max_tokens: int):
    encoded_prompts = [tokenizer.encode(prompt, bos=True, eos=False) for prompt in prompts]

    prompt_lens = [len(x) for x in encoded_prompts]
    min_prompt_len = min(prompt_lens)
    max_prompt_len = max(prompt_lens)

    input_tokens = torch.full(
        (len(prompts), max_prompt_len),
        tokenizer._model.pad_id(),
        dtype=torch.long,
        device=DEVICE,
    )
    for i, encoded in enumerate(encoded_prompts):
        input_tokens[i, : len(encoded)] = torch.tensor(encoded).to(input_tokens)
    input_mask = input_tokens != tokenizer._model.pad_id()

    # prefill as far as the shortest prompt, so every row sits at the same position
    positions = torch.arange(0, min_prompt_len).to(DEVICE)
    logits = model.forward(input_tokens[:, :min_prompt_len], positions)
    logprobs = nn.functional.log_softmax(logits, dim=-1)

    # decode
    generated = []

    cur_pos = min_prompt_len
    for _ in range(max_tokens):
        next_token = torch.argmax(logprobs[:, -1, :], dim=-1)
        if cur_pos < input_mask.shape[1]:
            # longer prompts are still being read: feed their next prompt token, not the prediction
            next_token = torch.where(input_mask[:, cur_pos], input_tokens[:, cur_pos], next_token)

        generated.append(next_token[:, None])

        logits = model.forward(next_token[:, None], torch.LongTensor([cur_pos]).to(next_token))
        logprobs = nn.functional.log_softmax(logits, dim=-1)
        cur_pos += 1

    res = []
    if max_tokens > 0:
        generated = torch.cat(generated, 1)

        for i, x in enumerate(encoded_prompts):
            res.append(tokenizer.decode(x[:min_prompt_len] + generated[i].tolist()))

    return res

In [ ]:
def demo(max_tokens: int = 100):
    mistral_tokenizer: MistralTokenizer = MistralTokenizer.from_file(f"{WEIGHTS_DIR}/tokenizer.model.v3")
    tokenizer: Tokenizer = mistral_tokenizer.instruct_tokenizer.tokenizer

    transformer = Transformer.from_folder(Path(WEIGHTS_DIR), max_batch_size=1, device=DEVICE)

    res = generate(
        [
            "What are transformers? "
        ],
        transformer,
        tokenizer,
        max_tokens=max_tokens,
    )
    for x in res:
        print(x)

In [ ]:
demo(100)

## 13. Next: make it efficient

This model is the `naive` rung of VELLUM v2's optimization ladder. `vellum/mistral/engine.py` drives these same modules and weights through one optimization at a time (last-position logits, no host syncs, grouped-query decode, fused attention, CUDA graphs), and `scripts/gcp_run.sh` benchmarks every rung against Hugging Face Transformers and vLLM on an L4. `tests/test_notebook_model.py` checks that the cells above build exactly the model the engine benchmarks.
